<a href="https://colab.research.google.com/github/tharunika-19/Machine_Learning/blob/main/cyclone_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_csv("Historical_Tropical_Storm_Tracks.csv")
print(df.columns.tolist())
print(df.shape)
print(df.head(2))

['index', 'FID', 'YEAR', 'MONTH', 'DAY', 'AD_TIME', 'BTID', 'NAME', 'LAT', 'LONG', 'WIND_KTS', 'PRESSURE', 'CAT', 'BASIN', 'Shape_Leng']
(37946, 15)
   index     FID    YEAR  MONTH  DAY AD_TIME   BTID      NAME   LAT   LONG  \
0      0  2001.0  1957.0    8.0  8.0   1800Z   63.0  NOTNAMED  22.5 -140.0   
1      1  2002.0  1961.0   10.0  3.0   1200Z  116.0   PAULINE  22.1 -140.2   

   WIND_KTS  PRESSURE CAT            BASIN  Shape_Leng  
0      50.0       0.0  TS  Eastern Pacific    1.140175  
1      45.0       0.0  TS  Eastern Pacific    1.166190  


In [ ]:
print(df['CAT'].value_counts())
print(df['PRESSURE'].value_counts().head(10))
print(df['WIND_KTS'].describe())


CAT
TS    16793
TD     7832
H1     6511
H2     2644
H3     1510
H4     1070
L       705
E       468
SS      141
H5      139
W        79
SD       53
Name: count, dtype: int64
PRESSURE
0.0       23008
1005.0      980
1009.0      931
1000.0      879
1007.0      737
1006.0      713
1008.0      712
1010.0      626
1002.0      558
1004.0      524
Name: count, dtype: int64
count    37945.000000
mean        53.414574
std         25.355396
min         10.000000
25%         35.000000
50%         45.000000
75%         70.000000
max        165.000000
Name: WIND_KTS, dtype: float64


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import pickle

# Load fresh
df = pd.read_csv("Historical_Tropical_Storm_Tracks.csv")

# Rename
df = df.rename(columns={
    'WIND_KTS': 'wind_speed',
    'PRESSURE': 'pressure',
    'CAT': 'category',
    'LAT': 'latitude',
    'LONG': 'longitude'
})

# Fix pressure
df['pressure'] = df['pressure'].replace(0, np.nan)
df['pressure'] = df['pressure'].fillna(df['pressure'].median())

# Create severity
def get_severity(wind):
    if wind < 63: return 0
    elif wind < 90: return 1
    elif wind < 120: return 2
    else: return 3

df['severity'] = df['wind_speed'].apply(get_severity)

# Target based on wind speed only (no category leak)
df['cyclone_risk'] = (df['wind_speed'] >= 90).astype(int)
print(df['cyclone_risk'].value_counts())

# Use only lat/lon — no leaking columns
X = df[['latitude', 'longitude']]
y = df['cyclone_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))
with open("cyclone_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("cyclone_model.pkl saved!")

cyclone_risk
0    52021
1     7207
Name: count, dtype: int64
Accuracy: 0.8356407226067871
cyclone_model.pkl saved!
